In [ ]:
import infostop
import polars as pl
import yaml
from pyproj import Transformer
from detect_stops import filter_pings, cluster_pings, assign_stop_ids, create_stop_table

In [4]:
with open("../config.yaml") as fp:
    config = yaml.safe_load(fp)
with open("05_detect_stops_config.yaml") as fp:
    stop_detections_config = yaml.safe_load(fp)

In [20]:
PARTITION = 0  # valid between 0 and 255

In [ ]:
# configure infostop
params = stop_detections_config["infostop"]

model = infostop.Infostop(
    r1=params["r1"],
    r2=params["r2"],
    min_staying_time=params["min_staying_time"],
    max_time_between=params["max_time_between"],
    min_size=params["min_size"],
    distance_metric=params["distance_metric"],
)

# set up transformer for reprojection
transformer = Transformer.from_crs(4326, config["crs"])

In [ ]:
data = pl.read_parquet(
    f"{config['dirs']['output']}/partitioned/{PARTITION:02x}/pings.parquet"
)

In [ ]:
filtered = filter_pings(
    data, min_pings=stop_detections_config["filtering"]["min_pings_per_day"]
)
filtered = filtered.with_columns(
    pl.col("timestamp").dt.epoch(time_unit="s").alias("epoch")
)

In [ ]:
try:
    clustered = pl.read_parquet(
        f"{config['dirs']['output']}/clustered_pings/{PARTITION:02x}.parquet"
    )
except FileNotFoundError:
    # runs about 5m
    clustered = cluster_pings(filtered, model)
    clustered = assign_stop_ids(clustered)
    clustered.write_parquet(
        f"{config['dirs']['output']}/clustered_pings/{PARTITION:02x}.parquet"
    )


In [ ]:
stops = create_stop_table(clustered)
stops.write_parquet(f"{config['dirs']['output']}/stops/{PARTITION:02x}.parquet")

In [ ]:
stops